# 🩻 NaijaCXR-VLM: Evaluation on MIMIC-CXR Test Split
This notebook loads the fine-tuned **NaijaCXR-VLM** (`MedGemma-4B` with Domain-Adapted `SigLIP-SO400M` Vision Tower) and generates radiology reports for the **MIMIC-CXR test split** (`mimic_test.json`).

### Pipeline Overview:
1. **Environment Setup & Dependencies**
2. **Path & Google Drive Configuration**
3. **Model Reconstruction & Architecture Surgery** (Transplanting domain-adapted SigLIP, 729-token embedding resize, projector patch)
4. **Input Preparation Pipeline** (Token expansion & 384x384 image scaling)
5. **MIMIC-CXR Batch Inference Engine** (Generating reports for 792 test cases)
6. **Clinical & NLP Metrics Evaluation** (ROUGE, BLEU, BERTScore, RadGraph F1)
7. **Qualitative Visualization & Grad-CAM Explainability**


## 1. Environment Setup & Dependencies


In [ ]:
# Install latest Hugging Face transformers, PEFT, and quantization libraries
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U peft bitsandbytes accelerate

# Install evaluation packages (ROUGE, BLEU, BERTScore, RadGraph)
!pip install -q evaluate rouge_score bert_score radgraph

# Install imaging & utility libraries
!pip install -q opencv-python matplotlib pillow pandas tqdm


## 2. Mount Google Drive & Import Libraries


In [ ]:
import os
import sys
import json
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    SiglipVisionModel
)
from peft import PeftModel

# Mount Google Drive if running in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print("🔌 Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("✅ Google Drive already mounted.")


## 3. Configuration & Paths


In [ ]:
# =========================================================
# CONFIGURATION
# =========================================================
BASE_MODEL_ID = "google/medgemma-4b-it"
SIGLIP_ID = "google/siglip-so400m-patch14-384"

# Trained Model Paths on Google Drive (Adjust if saved elsewhere)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/NaijaCXR_Project"
TRAINED_MODEL_PATH = os.path.join(DRIVE_PROJECT_DIR, "medgemma_naijacxr_V2")
VISION_ADAPTER_PATH = os.path.join(DRIVE_PROJECT_DIR, "siglip_lora_epoch_15")

# Test Split & Images Paths
# Check multiple possible locations (Drive, /content/, or local)
possible_test_paths = [
    "/content/mimic_test.json",
    os.path.join(DRIVE_PROJECT_DIR, "mimic_test.json"),
    "mimic_test.json"
]
TEST_FILE_PATH = next((p for p in possible_test_paths if os.path.exists(p)), "mimic_test.json")

possible_image_dirs = [
    "/content/Seleccted_mimic_CXR_images",
    os.path.join(DRIVE_PROJECT_DIR, "Seleccted_mimic_CXR_images"),
    "Seleccted_mimic_CXR_images"
]
IMAGE_ROOT_DIR = next((p for p in possible_image_dirs if os.path.exists(p)), "Seleccted_mimic_CXR_images")

CSV_OUTPUT_PATH = "MIMIC_CXR_NaijaCXR_VLM_predictions.csv"
METRICS_OUTPUT_PATH = "MIMIC_CXR_evaluation_metrics.json"

# Architecture Constants (384px image size / 14 patch size = 27x27 = 729 tokens)
TARGET_IMG_SIZE = 384
TARGET_PATCH_SIZE = 14
TARGET_TOKENS = 729
OLD_TOKENS = 256

print(f"📁 Test JSON:       {TEST_FILE_PATH} (Exists: {os.path.exists(TEST_FILE_PATH)})")
print(f"📁 Image Directory: {IMAGE_ROOT_DIR} (Exists: {os.path.exists(IMAGE_ROOT_DIR)})")
print(f"📁 Model Adapter:   {TRAINED_MODEL_PATH}")
print(f"📁 Vision Adapter:  {VISION_ADAPTER_PATH}")


### Optional: Unpack Dataset Archives (if zipped on Drive)


In [ ]:
# If Seleccted_mimic_CXR_images is zipped in Drive, unzip it to /content
zip_path = "/content/drive/MyDrive/Seleccted_mimic_CXR_images.zip"
if os.path.exists(zip_path) and not os.path.exists("/content/Seleccted_mimic_CXR_images"):
    print("📦 Unzipping MIMIC-CXR Images to /content/ ...")
    !unzip -q {zip_path} -d /content/
    print("✅ Unzipped successfully.")


## 4. Re-assembling the NaijaCXR-VLM Model
We reconstruct the fine-tuned architecture:
1. Load 4-bit quantized base `MedGemma-4B-IT`
2. Transplant the Domain-Adapted `SigLIP` Vision Tower (with merged `siglip_lora_epoch_15`)
3. Resize 2D Positional Embeddings from 4096 (64x64) -> 729 (27x27)
4. Patch the Multi-Modal Projector (remove pooling, force 27x27 patch grid)
5. Load the trained VLM PEFT adapter weights (`medgemma_naijacxr_V2`)
6. Configure the `AutoProcessor` with 384x384 resize


In [ ]:
def load_evaluation_model():
    print("🏗️  Re-assembling NaijaCXR-VLM for Evaluation...")

    # A. Load Base Model in 4-bit NF4
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )
    print(f"   Loading Base Model: {BASE_MODEL_ID} ...")
    model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )

    # B. Transplant Domain-Adapted SigLIP Vision Tower
    print("   🔌 Swapping & Merging Domain-Adapted Vision Tower...")
    vision_tower = SiglipVisionModel.from_pretrained(SIGLIP_ID, torch_dtype=torch.bfloat16)
    if os.path.exists(VISION_ADAPTER_PATH):
        vision_tower = PeftModel.from_pretrained(vision_tower, VISION_ADAPTER_PATH)
        vision_tower = vision_tower.merge_and_unload()
    else:
        print(f"   ⚠️ Warning: Vision adapter not found at {VISION_ADAPTER_PATH}, using base SigLIP.")
    
    vision_tower.to("cuda")
    if hasattr(model, "vision_tower"):
        model.vision_tower = vision_tower
    elif hasattr(model.model, "vision_tower"):
        model.model.vision_tower = vision_tower

    # C. Resize Embeddings (4096 -> 729 tokens)
    print("   📐 Resizing Position Embeddings (4096 -> 729)...")
    model.config.mm_tokens_per_image = TARGET_TOKENS
    model.config.vision_config.image_size = TARGET_IMG_SIZE
    model.config.vision_config.num_image_tokens = TARGET_TOKENS
    model.config.vision_config.patch_size = TARGET_PATCH_SIZE

    for name, param in model.named_parameters():
        if 4096 in param.shape:
            if param.ndim == 3:
                t = param.transpose(1, 2).view(1, -1, 64, 64)
            else:
                t = param.T.unsqueeze(0).view(1, -1, 64, 64)
            new_t = F.interpolate(t.float(), size=(27, 27), mode='bicubic')
            if param.ndim == 3:
                new_param = new_t.flatten(2).transpose(1, 2)
            else:
                new_param = new_t.flatten(2).transpose(1, 2).squeeze(0)
            param.data = new_param.to(param.dtype).to(param.device)

    # D. Patch Projector
    print("   🔧 Patching Multi-Modal Projector...")
    projector = model.model.multi_modal_projector if hasattr(model.model, "multi_modal_projector") else model.multi_modal_projector
    projector.to(dtype=torch.bfloat16)

    # Remove Average Pooling
    if hasattr(projector, "avg_pool"):
        projector.avg_pool = nn.Identity()

    # Force Projector Grid Size (64 -> 27)
    count = 0
    for m in model.modules():
        if hasattr(m, "patches_per_image") and m.patches_per_image == 64:
            m.patches_per_image = 27
            count += 1
    print(f"   └── Updated {count} layers to 27x27 patch grid.")

    # E. Load Trained VLM Adapter Weights
    if os.path.exists(TRAINED_MODEL_PATH):
        print(f"   💾 Loading Trained VLM Adapter: {TRAINED_MODEL_PATH}")
        model = PeftModel.from_pretrained(model, TRAINED_MODEL_PATH)
    else:
        print(f"   ⚠️ Warning: Trained model path {TRAINED_MODEL_PATH} not found.")

    model.eval()

    # Load & Configure Processor
    processor_source = TRAINED_MODEL_PATH if os.path.exists(TRAINED_MODEL_PATH) else BASE_MODEL_ID
    processor = AutoProcessor.from_pretrained(processor_source)
    processor.image_processor.size = {"height": TARGET_IMG_SIZE, "width": TARGET_IMG_SIZE}
    processor.image_processor.crop_size = {"height": TARGET_IMG_SIZE, "width": TARGET_IMG_SIZE}
    processor.image_processor.do_resize = True

    print("✅ Model Assembly Complete.")
    return model, processor


## 5. Input Preparation & Generation Functions


In [ ]:
def prepare_inputs(text, image, processor, device="cuda"):
    """Format image and text tokens ensuring correct 729-token sequence length."""
    # 1. Standard processing
    inputs = processor(text=text, images=image, return_tensors="pt").to(device)

    # 2. Force Pixel Values Dimensions to 384x384 in bfloat16
    if inputs["pixel_values"].shape[2] != TARGET_IMG_SIZE:
        inputs["pixel_values"] = F.interpolate(
            inputs["pixel_values"],
            size=(TARGET_IMG_SIZE, TARGET_IMG_SIZE),
            mode='bilinear',
            align_corners=False
        )
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    # 3. Expand Image Tokens (256 -> 729)
    input_ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    has_token_types = "token_type_ids" in inputs
    token_types = inputs.get("token_type_ids")

    new_ids_list = []
    new_mask_list = []
    new_types_list = []

    for i in range(len(input_ids)):
        seq = input_ids[i].tolist()

        # Find image token block
        u, c = torch.unique(input_ids[i], return_counts=True)
        img_id = u[c == OLD_TOKENS]
        if len(img_id) > 0:
            img_id = img_id[0].item()
            start = seq.index(img_id)

            prefix = seq[:start]
            img_block = [img_id] * TARGET_TOKENS
            suffix = seq[start + OLD_TOKENS:]

            new_ids_list.append(prefix + img_block + suffix)
            new_mask_list.append([1] * len(prefix) + [1] * TARGET_TOKENS + [1] * len(suffix))

            if has_token_types:
                new_types_list.append([0] * len(prefix) + [1] * TARGET_TOKENS + [0] * len(suffix))
        else:
            new_ids_list.append(seq)
            new_mask_list.append(mask[i].tolist())
            if has_token_types:
                new_types_list.append(token_types[i].tolist())

    inputs["input_ids"] = torch.tensor(new_ids_list, device=device)
    inputs["attention_mask"] = torch.tensor(new_mask_list, device=device)
    if has_token_types:
        inputs["token_type_ids"] = torch.tensor(new_types_list, device=device)

    return inputs


def generate_report(model, processor, image_path, prompt_text="Describe the chest X-ray findings and impression."):
    """Generate a radiology report for a single image."""
    try:
        # Resolve full path if relative
        if not os.path.isabs(image_path):
            candidates = [
                image_path,
                os.path.join(IMAGE_ROOT_DIR, os.path.basename(image_path)),
                os.path.join("/content", image_path),
                os.path.join("/content/Seleccted_mimic_CXR_images", os.path.basename(image_path))
            ]
            actual_path = next((p for p in candidates if os.path.exists(p)), image_path)
        else:
            actual_path = image_path

        image = Image.open(actual_path).convert("RGB")
    except Exception as e:
        return f"[Error: Image loading failed - {str(e)}]"

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
    text_input = processor.apply_chat_template(messages, add_generation_prompt=True)

    inputs = prepare_inputs(text_input, image, processor, device="cuda")

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pixel_values=inputs["pixel_values"],
            token_type_ids=inputs["token_type_ids"] if "token_type_ids" in inputs else None,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1
        )

    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


## 6. Initialize Model & Run Single-Image Verification


In [ ]:
# Instantiate model and processor
model, processor = load_evaluation_model()

# Quick test on first sample from mimic_test.json
with open(TEST_FILE_PATH, 'r') as f:
    sample_data = json.load(f)

sample_img = sample_data[0]["messages"][0]["content"][0]["image"]
print(f"\n🧪 Running test inference on: {sample_img}")
sample_report = generate_report(model, processor, sample_img)
print("\n🩻 Generated Report:")
print("-" * 50)
print(sample_report)
print("-" * 50)


## 7. Batch Inference on MIMIC-CXR Test Split


In [ ]:
print(f"📂 Loading Test Data: {TEST_FILE_PATH}")
with open(TEST_FILE_PATH, "r") as f:
    test_items = json.load(f)

# Select 60 samples from the MIMIC-CXR test split
NUM_SAMPLES = 60
test_items = test_items[:NUM_SAMPLES]

print(f"🚀 Generating Reports for {len(test_items)} MIMIC-CXR test cases...")

results = []
for idx, item in enumerate(tqdm(test_items, desc="MIMIC-CXR Inference")):
    img_path = item["messages"][0]["content"][0]["image"]
    prompt = item["messages"][0]["content"][1]["text"]
    ground_truth = item["messages"][1]["content"][0]["text"]

    prediction = generate_report(model, processor, img_path, prompt_text=prompt)

    results.append({
        "sample_id": idx,
        "image_path": img_path,
        "ground_truth": ground_truth,
        "prediction": prediction
    })

    # Periodically checkpoint to CSV every 10 samples
    if (idx + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(CSV_OUTPUT_PATH, index=False)

# Final Save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv(CSV_OUTPUT_PATH, index=False)
print(f"\n💾 Successfully saved {len(df_results)} predictions to: {CSV_OUTPUT_PATH}")


## 8. Clinical & Natural Language Metrics Evaluation
We evaluate the generated reports across multiple standard medical NLG metrics:
- **ROUGE-1, ROUGE-2, ROUGE-L** (N-gram overlap & longest common subsequence)
- **BLEU-1 to BLEU-4** (Precision-based n-gram matching)
- **BERTScore (F1)** (Semantic embedding similarity)
- **RadGraph F1** (Clinical entity & relation graph overlap)


In [ ]:
import evaluate
from bert_score import score as bert_score_func
from radgraph import F1RadGraph

# Load saved predictions
print(f"📊 Loading predictions from {CSV_OUTPUT_PATH}...")
df_eval = pd.read_csv(CSV_OUTPUT_PATH)

# Filter out error outputs if any
valid_df = df_eval[~df_eval["prediction"].str.startswith("[Error:")].copy()
print(f"Evaluating {len(valid_df)} / {len(df_eval)} valid predictions...")

references = valid_df["ground_truth"].astype(str).tolist()
predictions = valid_df["prediction"].astype(str).tolist()

metrics_summary = {}

# 1. ROUGE Metrics
print("\n1️⃣ Computing ROUGE...")
rouge = evaluate.load("rouge")
rouge_res = rouge.compute(predictions=predictions, references=references)
metrics_summary["ROUGE-1"] = round(rouge_res["rouge1"], 4)
metrics_summary["ROUGE-2"] = round(rouge_res["rouge2"], 4)
metrics_summary["ROUGE-L"] = round(rouge_res["rougeL"], 4)
print(f"   ROUGE-1: {metrics_summary['ROUGE-1']}")
print(f"   ROUGE-2: {metrics_summary['ROUGE-2']}")
print(f"   ROUGE-L: {metrics_summary['ROUGE-L']}")

# 2. BLEU Metrics
print("\n2️⃣ Computing BLEU (1-4)...")
bleu = evaluate.load("bleu")
bleu_res = bleu.compute(predictions=predictions, references=references)
metrics_summary["BLEU"] = round(bleu_res["bleu"], 4)
for i, prec in enumerate(bleu_res["precisions"]):
    metrics_summary[f"BLEU-{i+1}"] = round(prec, 4)
print(f"   BLEU:    {metrics_summary['BLEU']}")
print(f"   BLEU-1:  {metrics_summary.get('BLEU-1', 0.0)}")
print(f"   BLEU-2:  {metrics_summary.get('BLEU-2', 0.0)}")
print(f"   BLEU-3:  {metrics_summary.get('BLEU-3', 0.0)}")
print(f"   BLEU-4:  {metrics_summary.get('BLEU-4', 0.0)}")

# 3. BERTScore
print("\n3️⃣ Computing BERTScore...")
try:
    P, R, F1 = bert_score_func(predictions, references, lang="en", verbose=False)
    metrics_summary["BERTScore_F1"] = round(F1.mean().item(), 4)
    print(f"   BERTScore F1: {metrics_summary['BERTScore_F1']}")
except Exception as e:
    print(f"   ⚠️ BERTScore failed: {e}")
    metrics_summary["BERTScore_F1"] = None

# 4. RadGraph F1 (Clinical Correctness)
print("\n4️⃣ Computing RadGraph F1 (Clinical Entity & Relation Matching)...")
try:
    f1radgraph = F1RadGraph(reward_level="partial")
    radgraph_results = f1radgraph(hyps=predictions, refs=references)
    radgraph_f1 = float(np.mean(radgraph_results[0]))
    metrics_summary["RadGraph_F1"] = round(radgraph_f1, 4)
    print(f"   RadGraph F1:  {metrics_summary['RadGraph_F1']}")
except Exception as e:
    print(f"   ⚠️ RadGraph calculation note: {e}")
    metrics_summary["RadGraph_F1"] = None

# Save metrics summary to JSON
with open(METRICS_OUTPUT_PATH, "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\n💾 Saved Metrics Summary to: {METRICS_OUTPUT_PATH}")


## 9. Final Results Summary


In [ ]:
# Display formatted summary table
summary_df = pd.DataFrame([metrics_summary]).T.reset_index()
summary_df.columns = ["Metric", "NaijaCXR-VLM (MIMIC-CXR Test Score)"]

print("=" * 60)
print("             🏆 FINAL EVALUATION REPORT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))
print("=" * 60)


## 10. Qualitative Analysis: Sample Report Comparisons


In [ ]:
def display_sample_comparisons(df, num_samples=3):
    """Display side-by-side chest X-rays with ground truth and generated reports."""
    samples = df.sample(min(num_samples, len(df)), random_state=42)

    for _, row in samples.iterrows():
        img_path = row["image_path"]
        if not os.path.isabs(img_path):
            candidates = [
                img_path,
                os.path.join(IMAGE_ROOT_DIR, os.path.basename(img_path)),
                os.path.join("/content", img_path),
                os.path.join("/content/Seleccted_mimic_CXR_images", os.path.basename(img_path))
            ]
            actual_path = next((p for p in candidates if os.path.exists(p)), img_path)
        else:
            actual_path = img_path

        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        try:
            img = Image.open(actual_path).convert("RGB")
            ax.imshow(img, cmap="gray")
        except Exception:
            ax.text(0.5, 0.5, f"Image not found:\n{img_path}", ha="center", va="center")
        ax.axis("off")
        ax.set_title(f"MIMIC-CXR: {os.path.basename(img_path)}")
        plt.show()

        print("📋 GROUND TRUTH REPORT:")
        print(row["ground_truth"])
        print("\n🤖 NAIJACXR-VLM PREDICTED REPORT:")
        print(row["prediction"])
        print("=" * 80)

display_sample_comparisons(df_results, num_samples=3)


## 11. Visual Grounding & Explainability (Grad-CAM)
This module generates attention heatmaps for specific clinical keywords (e.g. `effusion`, `opacity`, `cardiomegaly`, `edema`) on the chest X-ray.


In [ ]:
class VLMGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.forward_hook = self.target_layer.register_forward_hook(self._save_activations)
        self.backward_hook = self.target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        if isinstance(output, tuple):
            self.activations = output[0]
        else:
            self.activations = output

    def _save_gradients(self, module, grad_input, grad_output):
        if isinstance(grad_output, tuple):
            self.gradients = grad_output[0]
        else:
            self.gradients = grad_output

    def generate_heatmap(self, target_logit):
        self.model.zero_grad()
        target_logit.backward(retain_graph=True)

        if self.gradients is None or self.activations is None:
            return np.zeros((384, 384))

        grads = self.gradients.detach()
        acts = self.activations.detach()

        if grads.ndim == 3:
            grads = grads.squeeze(0)
        if acts.ndim == 3:
            acts = acts.squeeze(0)

        weights = grads.mean(dim=0)
        heatmap = (acts * weights).sum(dim=-1)
        heatmap = F.relu(heatmap)

        h_min, h_max = heatmap.min(), heatmap.max()
        if h_max > h_min:
            heatmap = (heatmap - h_min) / (h_max - h_min)
        else:
            heatmap = torch.zeros_like(heatmap)

        seq_len = heatmap.shape[0]
        grid_size = int(seq_len ** 0.5)
        if grid_size * grid_size != seq_len:
            for offset in [1, 2]:
                new_len = seq_len - offset
                new_grid = int(new_len ** 0.5)
                if new_grid * new_grid == new_len:
                    heatmap = heatmap[offset:]
                    grid_size = new_grid
                    break

        heatmap_2d = heatmap.view(1, 1, grid_size, grid_size)
        heatmap_resized = F.interpolate(
            heatmap_2d, size=(384, 384), mode='bilinear', align_corners=False
        ).squeeze()
        return heatmap_resized.cpu().numpy()

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()


def plot_gradcam_overlay(image_path, heatmap, keyword, alpha=0.45):
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    else:
        img = image_path
    img_np = np.array(img.resize((384, 384)))

    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    overlayed = cv2.addWeighted(img_np, 1 - alpha, heatmap_colored, alpha, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Original Chest X-Ray")
    axes[0].axis("off")

    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title(f"Grad-CAM Heatmap: '{keyword}'")
    axes[1].axis("off")

    axes[2].imshow(overlayed)
    axes[2].set_title(f"Grounded Localization: '{keyword}'")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
